# Manga bubble training
Run each code cell in order. See `training/README.md` for prerequisites and limits.

In [ ]:
# Select a GPU runtime first (Runtime > Change runtime type).
# This workflow is available after the training package merges to main.
TRAINING_REVISION = "main"
%cd /content
!git clone --depth 1 --branch {TRAINING_REVISION} https://github.com/ntu254/storyboard-nexus.git
%cd /content/storyboard-nexus/ai-service
!pip install -q -r training/requirements.txt

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")

In [ ]:
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add the HF_TOKEN Colab Secret before continuing.")

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os
import shutil
import subprocess
import sys

IMAGE_ROOT = Path("/content/drive/MyDrive/Manga109s/images")
DRIVE_RUNS = Path("/content/drive/MyDrive/Manga109s/training-runs")
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f"Missing Manga109s images at {IMAGE_ROOT}. Create the Drive shortcut first.")

WORK = Path("/content/manga-bubble-training")
ANNOTATIONS = WORK / "annotations"
DATASET = WORK / "dataset"
LOCAL_RUNS = WORK / "runs"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = LOCAL_RUNS / RUN_ID

def run(*args):
    subprocess.run([sys.executable, *args], check=True, env={**os.environ, "HF_TOKEN": HF_TOKEN})

In [ ]:
run("-m", "training.download_annotations", "--destination", str(ANNOTATIONS))

In [ ]:
run("-m", "training.prepare_dataset", str(ANNOTATIONS / "jsons"), str(IMAGE_ROOT), str(DATASET))

In [ ]:
run("-m", "training.train", str(DATASET / "dataset.yaml"), str(LOCAL_RUNS / "smoke"), "--epochs", "1", "--run-name", "smoke")

In [ ]:
run("-m", "training.train", str(DATASET / "dataset.yaml"), str(RUN_DIR))

In [ ]:
BEST = RUN_DIR / "weights" / "best.pt"
if not BEST.is_file():
    raise FileNotFoundError(f"Training did not produce {BEST}")
run("-m", "training.evaluate", str(BEST), str(DATASET / "dataset.yaml"), str(RUN_DIR / "evaluation"))

In [ ]:
RESULTS = RUN_DIR / "results.csv"
if not RESULTS.is_file():
    raise FileNotFoundError(f"Training did not produce {RESULTS}")
DRIVE_RUN = DRIVE_RUNS / RUN_ID
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
shutil.copy2(RESULTS, DRIVE_RUN / "results.csv")

In [ ]:
DRIVE_RUN = DRIVE_RUNS / RUN_ID
PREDICTIONS = RUN_DIR / "evaluation" / "predictions"
(DRIVE_RUN / "weights").mkdir(parents=True, exist_ok=True)
(DRIVE_RUN / "evaluation").mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST, DRIVE_RUN / "weights" / "best.pt")
shutil.copy2(DATASET / "dataset.yaml", DRIVE_RUN / "dataset.yaml")
shutil.copy2(DATASET / "manifest.json", DRIVE_RUN / "manifest.json")
shutil.copy2(RUN_DIR / "evaluation" / "test-metrics.json", DRIVE_RUN / "evaluation" / "test-metrics.json")
if PREDICTIONS.is_dir():
    shutil.copytree(PREDICTIONS, DRIVE_RUN / "evaluation" / "predictions", dirs_exist_ok=True)
print(f"Saved {DRIVE_RUN / 'weights' / 'best.pt'}")